In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from vitpol import ViT
import time
import torch.nn.functional as F


            
def main():

    model = ViT(
        img_size=224,
        patch_size=16,
        num_classes=10,
        in_channels=3,
        dim=384,
        depth=7,
        heads=6
    )

    batch_size = 32

    # Нормализация ImageNet
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # Аугментации для train 
    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # ← ЗАПЯТАЯ!
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        normalize
    ])

    # Для test/val
    transform_test = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    train_dataset = torchvision.datasets.Imagenette(
        root='../data',
        split='train',
        size='320px',
        download=False,
        transform=transform_train
    )

    test_dataset = torchvision.datasets.Imagenette(
        root='../data',
        split='val',
        size='320px',
        download=False,
        transform=transform_test
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    def train_epoch(model, loader, optimizer, criterion, device):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    def test_epoch(model, loader, criterion, device):
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), correct / total

    # Настройки цикла
    epochs = 50
    test_accs = []
    epoch_times = []  # Список для хранения времени каждой эпохи
    best_test_loss = float('inf')

    # === EARLY STOPPING ===
    patience = 7
    epochs_no_improve = 0

    log_file = open("training_log.txt", "w", encoding="utf-8")

    print(f"Starting training on {device}...")

    for epoch in range(epochs):
        start_time = time.time()  # Засекаем время начала эпохи
        
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = test_epoch(model, test_loader, criterion, device)
        scheduler.step()
        
        end_time = time.time()  # Засекаем время окончания
        epoch_duration = end_time - start_time
        epoch_times.append(epoch_duration)
        
        test_accs.append(test_acc)

        # Подготовка строки лога
        log_str = (f"Epoch {epoch + 1}/{epochs} | "
                   f"Time: {epoch_duration:.2f}s | "
                   f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f} | "
                   f"Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}\n")
        
        print(log_str, end="")
        log_file.write(log_str)
        log_file.flush()

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f"---> New best model saved! Loss: {test_loss:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement: {epochs_no_improve}/{patience}")

        # === EARLY STOPPING ===
        if epochs_no_improve >= patience:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

        # if (epoch + 1) % 10 == 0:
            # torch.save(model.state_dict(), f'vit_epoch_{epoch+1}.pth')

    # Итоговая статистика
    avg_time = sum(epoch_times) / len(epoch_times)
    total_time = sum(epoch_times)
    
    summary_str = (f"\n{'='*30}\n"
                   f"Training Complete!\n"
                   f"Total Time: {total_time:.2f}s ({total_time/60:.2f} min)\n"
                   f"Average Time per Epoch: {avg_time:.2f}s\n"
                   f"Best Test Accuracy: {max(test_accs):.4f}\n"
                   f"{'='*30}")
    
    print(summary_str)
    log_file.write(summary_str + "\n")
    log_file.close()
    

if __name__ == "__main__":
    main()

ViT(
  (patch_embed): PatchEmbedding(
    (proj): Conv2d(1, 128, kernel_size=(7, 7), stride=(7, 7))
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MixtureAttention(
        (to_qkv): Linear(in_features=128, out_features=384, bias=False)
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
    (1): TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, element